### Load the shared setup context

**Purpose:** Establish the configured catalog, schema, and runtime paths for governance work.

**Inputs:** Demo2 Olist setup parameters.

**Outputs:** Reusable catalog and schema variables for the following governance cells.

**Why it matters:** Governance checks must run against the same isolated development context as the data products.

In [0]:
%run ./00_setup

### Select and verify the Unity Catalog context

**Purpose:** Confirm the notebook is operating in the configured catalog and schema.

**Inputs:** `catalog` and `schema` from setup.

**Outputs:** An active SQL context for governance objects.

**Why it matters:** Access controls are meaningful only when applied to the intended development namespace.

In [0]:
# Cell 2 — Select the project namespace and verify the active catalog and schema.
spark.sql(f"USE CATALOG `{catalog}`").collect()
spark.sql(f"USE SCHEMA `{schema}`").collect()

display(
    spark.sql("""
        SELECT
            current_catalog() AS current_catalog,
            current_schema() AS current_schema
    """)
)

### Inspect the Gold customer surface

**Purpose:** Read the customer dimension that governance rules protect.

**Inputs:** The current Gold customer table.

**Outputs:** A validated customer DataFrame for policy checks.

**Why it matters:** This is the governed source for customer analysis and contains sensitive attributes.

In [0]:
# Cell 3 — Remove policies accidentally attached to the core Gold customer table.
base_customer_table = f"{catalog}.{schema}.gold_dim_customer"

policy_cleanup = [
    (
        "row filter",
        f"ALTER TABLE {base_customer_table} DROP ROW FILTER",
        ["NO_ROW_FILTER", "ROW_FILTER_NOT_FOUND", "does not have a row filter"]
    ),
    (
        "customer_id column mask",
        (
            f"ALTER TABLE {base_customer_table} "
            "ALTER COLUMN customer_id DROP MASK"
        ),
        ["NO_MASK", "MASK_NOT_FOUND", "does not have a mask", "no column mask"]
    )
]

for policy_name, statement, missing_markers in policy_cleanup:
    try:
        spark.sql(statement)
        print(f"Removed accidental {policy_name} from Gold customer table")
    except Exception as error:
        error_message = str(error).lower()

        if any(
            marker.lower() in error_message
            for marker in missing_markers
        ):
            print(f"Gold customer {policy_name} was already absent")
        else:
            raise

# Remove the older duplicate functions after detaching their policies.
spark.sql(
    f"DROP FUNCTION IF EXISTS {catalog}.{schema}.customer_state_filter"
)
spark.sql(
    f"DROP FUNCTION IF EXISTS {catalog}.{schema}.mask_customer_id"
)

### Build the governed customer view

**Purpose:** Project customer attributes into the secured analytical surface.

**Inputs:** Gold customer records and the configured governance policy.

**Outputs:** A customer view with controlled columns and state context.

**Why it matters:** Centralizing the governed projection prevents consumers from bypassing the intended access boundary.

In [0]:
# Cell 4 — Reset the current user's RLS mapping to the normal SP and RJ scope.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS
        {catalog}.{schema}.governance_customer_state_access (
            principal STRING,
            customer_state STRING,
            _created_at TIMESTAMP
        )
    USING DELTA
""")

spark.sql(f"""
    DELETE FROM
        {catalog}.{schema}.governance_customer_state_access
    WHERE principal = session_user()
""")

spark.sql(f"""
    INSERT INTO
        {catalog}.{schema}.governance_customer_state_access (
            principal,
            customer_state,
            _created_at
        )
    SELECT
        session_user(),
        customer_state,
        current_timestamp()
    FROM VALUES
        ('SP'),
        ('RJ')
    AS allowed_states(customer_state)
""")

display(
    spark.sql(f"""
        SELECT *
        FROM {catalog}.{schema}.governance_customer_state_access
        WHERE principal = session_user()
        ORDER BY customer_state
    """)
)

### Apply column-level governance

**Purpose:** Define the protected customer columns and their intended access behavior.

**Inputs:** The governed customer view and Unity Catalog policy statements.

**Outputs:** Applied governance metadata or access-control configuration.

**Why it matters:** The workflow demonstrates how sensitive customer data is separated from broad analytical access.

In [0]:
# Cell 5 — Create one reusable RLS function and two reusable CLS mask functions.
# Manual prerequisite: create the account group `olist_pii_readers` in Identity and access.
spark.sql(f"""
    CREATE OR REPLACE FUNCTION
        {catalog}.{schema}.rls_customer_state_filter(
            row_customer_state STRING
        )
    RETURN EXISTS (
        SELECT 1
        FROM {catalog}.{schema}.governance_customer_state_access a
        WHERE a.principal = session_user()
          AND (
              a.customer_state = row_customer_state
              OR a.customer_state = 'ALL'
          )
    )
""")

spark.sql(f"""
    CREATE OR REPLACE FUNCTION
        {catalog}.{schema}.mask_customer_identifier(
            original_value STRING
        )
    RETURN
        CASE
            WHEN original_value IS NULL THEN NULL
            WHEN is_account_group_member('olist_pii_readers')
                THEN original_value
            ELSE CONCAT(
                SUBSTRING(original_value, 1, 4),
                '...MASKED'
            )
        END
""")

spark.sql(f"""
    CREATE OR REPLACE FUNCTION
        {catalog}.{schema}.mask_customer_zip(
            original_value STRING
        )
    RETURN
        CASE
            WHEN original_value IS NULL THEN NULL
            WHEN is_account_group_member('olist_pii_readers')
                THEN original_value
            ELSE CONCAT(
                SUBSTRING(original_value, 1, 3),
                '**'
            )
        END
""")

print("RLS and CLS functions created")

### Configure row-level access scope

**Purpose:** Define the customer-state mapping used by row-level security.

**Inputs:** Customer state values and the project governance mapping.

**Outputs:** Access rows that determine which records a governed user can see.

**Why it matters:** Regional filtering demonstrates least-privilege access without copying the Gold table.

In [0]:
# Cell 6 — Unit-test the RLS decision and both masking functions before using them.
rls_function_test_df = spark.sql(f"""
    SELECT
        customer_state,
        {catalog}.{schema}.rls_customer_state_filter(
            customer_state
        ) AS can_current_user_see
    FROM VALUES
        ('SP'),
        ('RJ'),
        ('MG'),
        ('BA')
    AS test_states(customer_state)
""")

mask_function_test_df = spark.sql(f"""
    SELECT
        original_identifier,
        {catalog}.{schema}.mask_customer_identifier(
            original_identifier
        ) AS returned_identifier,
        original_zip,
        {catalog}.{schema}.mask_customer_zip(
            original_zip
        ) AS returned_zip,
        is_account_group_member(
            'olist_pii_readers'
        ) AS is_pii_reader
    FROM VALUES
        ('00012a2ce6f8dcda20d059ce98491703', '06273'),
        (CAST(NULL AS STRING), CAST(NULL AS STRING))
    AS test_values(original_identifier, original_zip)
""")

display(rls_function_test_df)
display(mask_function_test_df)

### Register reusable governance functions

**Purpose:** Create the row-filter and masking functions referenced by governed views.

**Inputs:** Current user identity, access mapping, and sensitive column values.

**Outputs:** Reusable Unity Catalog functions for RLS and CLS behavior.

**Why it matters:** Reusable functions keep policy behavior consistent across consumers.

In [0]:
# Cell 7 — Create the governed customer view by reusing the RLS and CLS functions.
spark.sql(f"""
    CREATE OR REPLACE VIEW
        {catalog}.{schema}.secure_gold_dim_customer
    AS
    SELECT
        c.customer_sk,
        {catalog}.{schema}.mask_customer_identifier(
            c.customer_id
        ) AS customer_id,
        {catalog}.{schema}.mask_customer_identifier(
            c.customer_unique_id
        ) AS customer_unique_id,
        {catalog}.{schema}.mask_customer_zip(
            c.customer_zip_code_prefix
        ) AS customer_zip_code_prefix,
        c.customer_city,
        c.customer_state,
        c.is_current
    FROM {catalog}.{schema}.gold_dim_customer c
    WHERE {catalog}.{schema}.rls_customer_state_filter(
        c.customer_state
    )
""")

print("Secure customer view created")

### Test policy decisions before publication

**Purpose:** Exercise RLS and masking logic with representative identities and values.

**Inputs:** Governance functions and sample customer contexts.

**Outputs:** Assertions or diagnostic results showing expected allow and mask behavior.

**Why it matters:** Policy tests catch governance regressions before secured views are published.

In [0]:
# Cell 8 — Validate that the secure view exposes only authorized and correctly masked data.
secure_customer_validation_df = spark.sql(f"""
    WITH user_context AS (
        SELECT
            is_account_group_member(
                'olist_pii_readers'
            ) AS is_pii_reader
    ),

    checks AS (
        SELECT
            COUNT(*) AS visible_rows,
            SORT_ARRAY(
                COLLECT_SET(c.customer_state)
            ) AS visible_states,
            u.is_pii_reader,

            SUM(
                CASE
                    WHEN NOT {catalog}.{schema}.rls_customer_state_filter(
                        c.customer_state
                    )
                    THEN 1
                    ELSE 0
                END
            ) AS unauthorized_rows,

            SUM(
                CASE
                    WHEN u.is_pii_reader = FALSE
                         AND (
                             c.customer_id NOT LIKE '%...MASKED'
                             OR c.customer_unique_id NOT LIKE '%...MASKED'
                             OR c.customer_zip_code_prefix NOT LIKE '%**'
                         )
                    THEN 1

                    WHEN u.is_pii_reader = TRUE
                         AND (
                             c.customer_id LIKE '%...MASKED'
                             OR c.customer_unique_id LIKE '%...MASKED'
                             OR c.customer_zip_code_prefix LIKE '%**'
                         )
                    THEN 1

                    ELSE 0
                END
            ) AS cls_violations

        FROM {catalog}.{schema}.secure_gold_dim_customer c
        CROSS JOIN user_context u
        GROUP BY u.is_pii_reader
    )

    SELECT
        *,
        CASE
            WHEN unauthorized_rows = 0
                 AND cls_violations = 0
            THEN 'PASS'
            ELSE 'FAIL'
        END AS status
    FROM checks
""")

display(secure_customer_validation_df)

display(
    spark.table(
        f"{catalog}.{schema}.secure_gold_dim_customer"
    ).limit(10)
)

### Publish the governed customer view

**Purpose:** Create or refresh the secured customer view used by dashboard and analyst queries.

**Inputs:** Gold customer data plus the registered RLS and CLS functions.

**Outputs:** `secure_gold_dim_customer` or the project’s governed customer surface.

**Why it matters:** Downstream consumers receive policy-aware data without changing the underlying Gold fact.

In [0]:
# Cell 9 — Recreate a small synthetic table for safe native-policy demonstrations.
spark.sql(f"""
    DROP TABLE IF EXISTS
        {catalog}.{schema}.governance_customer_policy_demo
""")

spark.sql(f"""
    CREATE TABLE
        {catalog}.{schema}.governance_customer_policy_demo
    USING DELTA
    AS SELECT *
    FROM VALUES
        (
            'customer_sp_001',
            'unique_sp_001',
            '01001',
            'Sao Paulo',
            'SP'
        ),
        (
            'customer_rj_001',
            'unique_rj_001',
            '20001',
            'Rio de Janeiro',
            'RJ'
        ),
        (
            'customer_mg_001',
            'unique_mg_001',
            '30001',
            'Belo Horizonte',
            'MG'
        ),
        (
            'customer_ba_001',
            'unique_ba_001',
            '40001',
            'Salvador',
            'BA'
        )
    AS demo(
        customer_id,
        customer_unique_id,
        customer_zip_code_prefix,
        customer_city,
        customer_state
    )
""")

print("Native-policy demonstration table recreated with four rows")

### Validate governed output behavior

**Purpose:** Verify that the secure view returns authorized rows and masks protected values as designed.

**Inputs:** The published governed customer view and policy functions.

**Outputs:** Validation counts and sample policy results.

**Why it matters:** Governance is part of the Job’s release contract, not an optional presentation step.

In [0]:
# Cell 10 — Attach native RLS and native CLS policies directly to the demo table.
demo_table = (
    f"{catalog}.{schema}.governance_customer_policy_demo"
)

spark.sql(f"""
    ALTER TABLE {demo_table}
    SET ROW FILTER
        {catalog}.{schema}.rls_customer_state_filter
    ON (customer_state)
""")

spark.sql(f"""
    ALTER TABLE {demo_table}
    ALTER COLUMN customer_id
    SET MASK {catalog}.{schema}.mask_customer_identifier
""")

spark.sql(f"""
    ALTER TABLE {demo_table}
    ALTER COLUMN customer_unique_id
    SET MASK {catalog}.{schema}.mask_customer_identifier
""")

spark.sql(f"""
    ALTER TABLE {demo_table}
    ALTER COLUMN customer_zip_code_prefix
    SET MASK {catalog}.{schema}.mask_customer_zip
""")

print("Native RLS and CLS policies attached to demo table")

### Demonstrate policy attachment safely

**Purpose:** Prepare the small demonstration relation used to show native policy attachment.

**Inputs:** Representative synthetic governance rows.

**Outputs:** A non-production demonstration object for policy verification.

**Why it matters:** Separating the demo object protects the production-style Gold tables from exploratory policy changes.

In [0]:
# Cell 11 — Validate native table policies and preview the caller-specific result.
native_policy_validation_df = spark.sql(f"""
    WITH user_context AS (
        SELECT
            is_account_group_member(
                'olist_pii_readers'
            ) AS is_pii_reader
    ),

    expected AS (
        SELECT
            COUNT(*) AS expected_visible_rows
        FROM VALUES
            ('SP'),
            ('RJ'),
            ('MG'),
            ('BA')
        AS source_states(customer_state)
        WHERE {catalog}.{schema}.rls_customer_state_filter(
            customer_state
        )
    ),

    checks AS (
        SELECT
            COUNT(*) AS actual_visible_rows,
            SORT_ARRAY(
                COLLECT_SET(d.customer_state)
            ) AS visible_states,
            u.is_pii_reader,

            SUM(
                CASE
                    WHEN NOT {catalog}.{schema}.rls_customer_state_filter(
                        d.customer_state
                    )
                    THEN 1
                    ELSE 0
                END
            ) AS unauthorized_rows,

            SUM(
                CASE
                    WHEN u.is_pii_reader = FALSE
                         AND (
                             d.customer_id NOT LIKE '%...MASKED'
                             OR d.customer_unique_id NOT LIKE '%...MASKED'
                             OR d.customer_zip_code_prefix NOT LIKE '%**'
                         )
                    THEN 1

                    WHEN u.is_pii_reader = TRUE
                         AND (
                             d.customer_id LIKE '%...MASKED'
                             OR d.customer_unique_id LIKE '%...MASKED'
                             OR d.customer_zip_code_prefix LIKE '%**'
                         )
                    THEN 1

                    ELSE 0
                END
            ) AS cls_violations

        FROM {catalog}.{schema}.governance_customer_policy_demo d
        CROSS JOIN user_context u
        GROUP BY u.is_pii_reader
    )

    SELECT
        e.expected_visible_rows,
        c.*,
        CASE
            WHEN c.actual_visible_rows = e.expected_visible_rows
                 AND c.unauthorized_rows = 0
                 AND c.cls_violations = 0
            THEN 'PASS'
            ELSE 'FAIL'
        END AS status
    FROM expected e
    CROSS JOIN checks c
""")

display(
    spark.table(
        f"{catalog}.{schema}.governance_customer_policy_demo"
    )
)

display(native_policy_validation_df)

### Verify native governance behavior

**Purpose:** Query the demonstration object and verify native row filtering and column masking.

**Inputs:** The demo table and attached policies.

**Outputs:** Evidence that policy predicates and masks are active.

**Why it matters:** The final governance result documents the protection expected by downstream users.